In [1]:
from pathlib import Path
from typing import List
import pandas as pd
import sys
import yaml
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from data_classes import timepoint, video, roi,neuron
from utils.io_utils import load_models 
from pipeline.io_handlers import save_filtered_suite2p, visualize_neuron_groups

In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
        
models = load_models(config)

Loaded ROI classifier from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\trained_models_notebooks\roi_classifier\roi_classifier_20260108_125706.joblib
Loaded Spike classifier from C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\trained_models_notebooks\spike_classifier\spike_classifier_20260109_145607.joblib


c:\Users\mzinn1\AppData\Local\anaconda3\envs\gcamp\lib\site-packages\_distutils_hack\__init__.py:30: UserWarning: Setuptools is replacing distutils. Support for replacing an already imported distutils is deprecated. In the future, this condition will fail. Register concerns at https://github.com/pypa/setuptools/issues/new?template=distutils-deprecation.yml
  warnings.warn(
Failed to load CASCADE model: [Errno 2] No such file or directory: 'Cascade/Pretrained_models\\available_models.yaml'


2.10.0


Failed to load Cascade model from Cascade/Pretrained_models with name Global_EXC_30Hz_smoothing100ms_high_noise: [Errno 2] No such file or directory: 'Cascade/Pretrained_models\\available_models.yaml'


Failed to load Cascade model from Cascade/Pretrained_models with name Global_EXC_30Hz_smoothing100ms_high_noise: [Errno 2] No such file or directory: 'Cascade/Pretrained_models\\available_models.yaml'


In [3]:
timepoint_path = Path(r"C:\Users\mzinn1\Desktop\test_notebook\week 2")
timepoint_obj = timepoint.Timepoint(timepoint_path)   
video_paths = [f for f in timepoint_path.iterdir() if f.is_dir()]
video_objs = [video.Video(vp, vp / "suite2p/plane0") for vp in video_paths]


In [4]:
def roi_aggregation(suite2p_data: dict, n_rois: int) -> list[roi.ROI]:  
    all_rois = []
    for i in range(n_rois):
        roi_obj = roi.ROI(
            index=i,
            f_trace=suite2p_data['F'][i],
            stats=suite2p_data['stat'][i] if 'stat' in suite2p_data else None,
            fneu=suite2p_data['Fneu'][i] if 'Fneu' in suite2p_data else None,
        )
        all_rois.append(roi_obj)
    return all_rois

In [5]:
for video in video_objs:
    _, norm_sm_f, norm_sg_f, _ = video.process_fluorescence_traces()
    n_rois, n_frames = norm_sm_f.shape
    all_rois = roi_aggregation(video.suite2p_data, n_rois)
    good_rois, bad_rois, good_roi_masks = video.filter_rois(all_rois, models["roi_classifier"])
    filtered_suite2p_path = save_filtered_suite2p(video_path=video.path, good_roi_mask=good_roi_masks, 
                                                suite2p_data=video.suite2p_data)
    video.neurons = [neuron.Neuron(roi, i, fs = video.suite2p_data['ops']['fs']) for i, roi in enumerate(good_rois)]
    print(f"Video {video.path.name} processed. Filtered suite2p data saved to {Path(*filtered_suite2p_path.parts[-4:])}")
    print(f"\tGood ROIs: {len(good_rois)}, Bad ROIs: {len(bad_rois)}")

Video 2-1 processed. Filtered suite2p data saved to week 2\2-1\filtered_suite2p\plane0
	Good ROIs: 58, Bad ROIs: 90
Video 2-1_5um_1m processed. Filtered suite2p data saved to week 2\2-1_5um_1m\filtered_suite2p\plane0
	Good ROIs: 45, Bad ROIs: 82
Video 2-2 processed. Filtered suite2p data saved to week 2\2-2\filtered_suite2p\plane0
	Good ROIs: 53, Bad ROIs: 105
Video 2-2_5um_3m processed. Filtered suite2p data saved to week 2\2-2_5um_3m\filtered_suite2p\plane0
	Good ROIs: 43, Bad ROIs: 121
Video 2-3 processed. Filtered suite2p data saved to week 2\2-3\filtered_suite2p\plane0
	Good ROIs: 78, Bad ROIs: 149
Video 2-3_5um_5m processed. Filtered suite2p data saved to week 2\2-3_5um_5m\filtered_suite2p\plane0
	Good ROIs: 113, Bad ROIs: 233
Video 2-4 processed. Filtered suite2p data saved to week 2\2-4\filtered_suite2p\plane0
	Good ROIs: 32, Bad ROIs: 34
Video 2-4_50um_1m processed. Filtered suite2p data saved to week 2\2-4_50um_1m\filtered_suite2p\plane0
	Good ROIs: 22, Bad ROIs: 99


In [6]:
for video in video_objs:
    spike_feats_df, spike_mask = video.get_all_spike_features(models["spike_classifier"])
    video.filter_all_spikes(spike_mask)
    spike_summaries_per_neuron = video.get_spike_statistics()
    video.summary_df = pd.DataFrame.from_dict(spike_summaries_per_neuron, orient='index')
    print(f"Spike statistics computed for video {video.path.name}.")
    candidate_spikes = len(spike_mask)
    positive_spikes = spike_mask.sum()
    print(f"\tClassified {positive_spikes} / {candidate_spikes} candidate spikes as true spikes.")

Spike statistics computed for video 2-1.
	Classified 1094 / 1702 candidate spikes as true spikes.
Spike statistics computed for video 2-1_5um_1m.
	Classified 989 / 1429 candidate spikes as true spikes.
Spike statistics computed for video 2-2.
	Classified 1095 / 1751 candidate spikes as true spikes.
Spike statistics computed for video 2-2_5um_3m.
	Classified 935 / 1449 candidate spikes as true spikes.
Spike statistics computed for video 2-3.
	Classified 1348 / 2175 candidate spikes as true spikes.
Spike statistics computed for video 2-3_5um_5m.
	Classified 2343 / 3493 candidate spikes as true spikes.
Spike statistics computed for video 2-4.
	Classified 492 / 846 candidate spikes as true spikes.
Spike statistics computed for video 2-4_50um_1m.
	Classified 405 / 698 candidate spikes as true spikes.


In [8]:
for video in video_objs:
    grouping_config = config["grouping"]["sttc"]
    time_window = grouping_config["time_window"]
    distance_threshold = grouping_config["distance_threshold"]

    grouping_stats, sttc_groups = video.get_group_summary(time_window, distance_threshold)
    video.grouping_stats = grouping_stats
    img_size = video.suite2p_data['ops']['Ly'], video.suite2p_data['ops']['Lx']
    if len(sttc_groups) > 0:
        visualize_neuron_groups(sttc_groups, video.suite2p_data["stat"], img_size, video.path, config_label="sttc")
    

In [17]:
for video in video_objs:
    per_neuron_summary = video.summary_df
    grouping_summary = None
    if len(video.grouping_stats["combined_stats"]) > 0:
        grouping_summary = pd.DataFrame.from_dict(video.grouping_stats, orient='columns')
        
    metrics_path = video.path/ f"metrics/{video.path.name}_metrics.xlsx" 
    metrics_path.parent.mkdir(parents=True, exist_ok=True)

    with pd.ExcelWriter(metrics_path) as writer:
        per_neuron_summary.to_excel(writer, sheet_name="per_neuron_summary")
        if grouping_summary is not None:
            grouping_summary.to_excel(writer, sheet_name="grouping_summary")
    